# Analisis Faktor Penentu Utama (Feature Importance) dengan Random Forest

Notebook ini bertujuan untuk mengidentifikasi dan memeringkat faktor-faktor sosio-ekonomi yang paling berpengaruh dalam menentukan apakah suatu provinsi memiliki tingkat NEET 'Tinggi' atau 'Rendah'. Kita akan menggunakan model klasifikasi **Random Forest** untuk mengekstrak 'feature importance' dari data.

## 1. Persiapan Data

Langkah pertama adalah memuat data, membersihkannya, dan yang terpenting, membuat variabel target kategoris ('NEET Tinggi' / 'NEET Rendah') berdasarkan perbandingan dengan rata-rata nasional (20,31%).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

# Membaca data dari file CSV
try:
    df = pd.read_csv('Data/Klaster Provinsi.csv')
    print("Data berhasil dimuat dari 'Data/Klaster Provinsi.csv'.")
    
    # Mengganti nama kolom agar lebih mudah digunakan
    df.rename(columns={
        'Tingkat NEET (%) (2024)': 'NEET',
        'TPT (%) (Feb 2023)': 'TPT',
        'RLS (Tahun) (2024)': 'RLS',
        'PDRB per Kapita (Juta Rp) (2023)': 'PDRB_per_Kapita'
    }, inplace=True)
    
    # Menangani data yang hilang dan memilih variabel
    features = ['TPT', 'RLS', 'PDRB_per_Kapita']
    df_model = df.dropna(subset=['NEET'] + features).copy()

    # Membuat variabel target kategoris
    national_average_neet = 20.31
    df_model['Kategori NEET'] = np.where(df_model['NEET'] > national_average_neet, 'Tinggi', 'Rendah')

    # Mendefinisikan variabel X (fitur) dan Y (target)
    X = df_model[features]
    Y = df_model['Kategori NEET']
    
    print("\nData siap untuk analisis Random Forest.")
    display(df_model[['Provinsi', 'NEET', 'Kategori NEET']].head())
    
except FileNotFoundError:
    print("Error: File 'Data/Klaster Provinsi.csv' tidak ditemukan.")

Data berhasil dimuat dari 'Data/Klaster Provinsi.csv'.

Data siap untuk analisis Random Forest.


,Provinsi,NEET,Kategori NEET
0,Aceh,28.56,Tinggi
1,Sumatera Utara,19.78,Rendah
2,Sumatera Barat,21.31,Tinggi
3,Riau,21.79,Tinggi
4,Jambi,20.71,Tinggi


## 2. Pelatihan Model dan Ekstraksi Feature Importance

Kita akan melatih model Random Forest Classifier pada data. Setelah model dilatih, kita dapat langsung mengekstrak skor 'importance' atau 'pentingnya' setiap fitur.

In [2]:
# Inisialisasi dan pelatihan model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, Y)

# Ekstraksi feature importances
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Faktor': features,
    'Skor Pentingnya': importances
}).sort_values(by='Skor Pentingnya', ascending=False)

print("Skor Pentingnya Setiap Faktor:")
display(feature_importance_df)

Skor Pentingnya Setiap Faktor:


,Faktor,Skor Pentingnya
1,RLS,0.362453
0,TPT,0.336255
2,PDRB_per_Kapita,0.301292


## 3. Visualisasi dan Interpretasi Hasil

Hasilnya paling baik ditampilkan dalam bentuk bar chart sederhana yang menunjukkan peringkat setiap faktor. Ini memberikan jawaban yang jelas dan langsung mengenai 'apa yang paling penting'.

In [3]:
# Visualisasi hasil dengan Plotly
fig = px.bar(
    feature_importance_df.sort_values(by='Skor Pentingnya', ascending=True),
    x='Skor Pentingnya',
    y='Faktor',
    orientation='h',
    title='<b>Peringkat Faktor Penentu Tingkat NEET Provinsi</b>',
    text=feature_importance_df.sort_values(by='Skor Pentingnya', ascending=True)['Skor Pentingnya'].apply(lambda x: f'{x:.2%}'),
    labels={'Skor Pentingnya': 'Skor Pentingnya (Importance)', 'Faktor': 'Faktor Sosio-Ekonomi'}
)
fig.update_layout(template='plotly_white')
fig.show()

# Interpretasi
most_important_factor = feature_importance_df.iloc[0]
print(f"\nInterpretasi: Berdasarkan model Random Forest, faktor yang menjadi penentu utama adalah '{most_important_factor['Faktor']}' ",
      f"dengan skor pentingnya sebesar {most_important_factor['Skor Pentingnya']:.2%}. ",
      "Ini berarti, secara nasional, variabel inilah yang paling kuat dalam membedakan antara provinsi dengan masalah NEET tinggi dan rendah.")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.





Interpretasi: Berdasarkan model Random Forest, faktor yang menjadi penentu utama adalah 'RLS'  dengan skor pentingnya sebesar 36.25%.  Ini berarti, secara nasional, variabel inilah yang paling kuat dalam membedakan antara provinsi dengan masalah NEET tinggi dan rendah.
